# 07 · Stable Audio 3: 구간 인페인팅 (음악 + 텍스트 → 음악)

**Stable Audio 3 Medium** (Stability AI, 2026). 텍스트 인코더(T5Gemma)가 프롬프트를 숫자로 바꾸고, 확산 트랜스포머(DiT)가 노이즈를 8단계에 걸쳐 음악의 압축 표현(latent)으로 다듬은 뒤, 오디오 디코더(SAME-L)가 44.1 kHz 스테레오로 되돌립니다. 이 노트북은 공식 CPU 런타임(LiteRT)을 써서 **GPU와 로그인 없이** 돌아갑니다.

**인페인팅**은 곡의 한 구간만 노이즈에서 다시 만들고, 구간 밖은 문맥으로 씁니다. 기본 예제는 잔잔한 피아노 24초 중 **6~18초를 웅장한 오케스트라**로 바꿉니다. 원본 파일은 바꾸지 않습니다.

- 입력 → 출력: 음악 오디오 + 텍스트 + 구간 → 그 구간만 바뀐 음악 오디오
- 소요 시간: 설치 1분 + 가중치 내려받기 2~3분(처음 한 번, 06과 같은 파일) + CPU 2코어 기준 피아노 생성 약 40초, 인페인팅 약 65초
- GPU 불필요
- 논문: Evans et al., "Stable Audio 3," 2026 (arXiv:2605.17991) · 코드: https://github.com/Stability-AI/stable-audio-3

In [ ]:
#@title 0. 준비: 수업 자료 받기와 모델 폴더 정하기 (실행만 하면 됩니다)
#@markdown 모델 가중치는 모두 `models` 폴더 하나에 받습니다. 기본 위치는 `/content`이고 런타임을 끄면 지워집니다. 체크하면 Google Drive의 `CT-AI_Microdegree/` 아래에 `models`와 `outputs`를 두어 다음 실행 때 다시 받지 않고, 다른 노트북의 결과도 이어서 쓸 수 있습니다.
KEEP_IN_DRIVE = False  #@param {type:"boolean"}
import os, sys, glob, time, json, subprocess
_cands = [os.environ.get('CTAI_REPO_DIR'), os.path.abspath(os.path.join(os.getcwd(), '..', '..')), '/content/CT-AI_Microdegree']
REPO = next((c for c in _cands if c and os.path.isfile(os.path.join(c, 'colab', 'ctai.py'))), None)
if REPO is None:
    REPO = '/content/CT-AI_Microdegree'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/dabinkim0/CT-AI_Microdegree', REPO], check=True)
sys.path.insert(0, os.path.join(REPO, 'colab'))
import ctai
P = ctai.setup(keep_in_drive=KEEP_IN_DRIVE)
AUDIO_DIR, ASSET_DIR, MODELS_DIR, OUT_DIR, IN_COLAB = P['AUDIO_DIR'], P['ASSET_DIR'], P['MODELS_DIR'], P['OUT_DIR'], P['IN_COLAB']
print('예시 음원:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(AUDIO_DIR, '*.wav'))))


In [ ]:
#@title 설치와 모델 준비 (약 1분, 가중치는 처음 실행할 때 내려받음)
#@markdown `medium`(1.4 B, 내려받기 약 3.2 GB + 첫 실행 때 만드는 CPU 캐시 약 2.3 GB)이 기본입니다. 느리면 `sm-music`(작은 음악 모델, 약 1.2 GB, 최대 120초)을 고르세요.
MODEL = 'medium'  #@param ["medium", "sm-music"]
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ai-edge-litert>=2.2.0', 'sentencepiece', 'soundfile',
                    'huggingface_hub', 'librosa', 'scipy', 'matplotlib'], capture_output=True, text=True)
print(r.stderr[-600:] if r.returncode else '')
import sa3
sa3.prepare(MODELS_DIR, OUT_DIR, dit=MODEL, decoder='same-l' if MODEL == 'medium' else 'same-s')
print('설치 OK')

In [ ]:
#@title 원본과 목표 스타일
SOURCE = '피아노 생성 (24초)'  #@param ["피아노 생성 (24초)", "예시 (재즈 Vibe Ace, 앞 24초)", "업로드"]
START_SECONDS = 6.0  #@param {type:"number"}
END_SECONDS = 18.0   #@param {type:"number"}
TARGET_STYLE = 'orchestral_ensemble'  #@param ["orchestral_ensemble", "drum_and_bass", "heavy_metal"]
SOURCE_PROMPT = ("Instrumental solo felt piano, slow sparse delicate notes, peaceful intimate acoustic ballad, "
                 "gentle dynamics, no drums, no percussion, no synthesizers, no vocals.")
STYLE_PROMPTS = {
    "orchestral_ensemble": "Instrumental cinematic orchestral ensemble, 100 BPM, full acoustic orchestra with lush layered violin and viola sections, rhythmic cellos and double basses, soaring French horns and brass, lyrical woodwinds, timpani rolls and cymbal swells, bold sweeping melody, powerful dramatic crescendo, rich concert hall acoustics, full ensemble rather than solo piano, no synthesizers, no vocals.",
    "drum_and_bass": "Instrumental aggressive drum and bass, 174 BPM, rapid chopped breakbeats, heavy distorted reese bass, sharp snares, bright high-speed hi-hats, intense energetic electronic drop, no vocals.",
    "heavy_metal": "Instrumental heavy metal, fast double-kick drums, heavily distorted electric guitar power-chord riffs, aggressive bass guitar, crashing cymbals, loud energetic full band, no vocals.",
}
EDIT_PROMPT = STYLE_PROMPTS[TARGET_STYLE]
SEED = 73
STEPS = 8
CFG = 4.0   # 원곡 성격을 밀어내려고 negative prompt를 쓰므로 1보다 커야 함
NEGATIVE_PROMPT = "gentle solo piano, quiet acoustic ballad, ambient pads, soft sparse music, vocals, speech"
CROSSFADE_SECONDS = 0.10

if SOURCE.startswith('피아노'):
    source = sa3.generate(SOURCE_PROMPT, seconds=24, seed=72)
elif SOURCE.startswith('예시'):
    source = os.path.join(AUDIO_DIR, 'jazz_vibe_ace_60s.wav')
else:
    from google.colab import files
    up = files.upload(); source = os.path.abspath(list(up.keys())[0])
MAX_INPUT_SECONDS = 24   # CPU 시간을 아끼려고 앞 24초만 씀
print('원본:', source)
sa3.plot_audio(source, region=(START_SECONDS, END_SECONDS))
_ = sa3.show_audio(source)

## 구간 재생성

`inpaint`는 한 번의 호출로 ① 입력을 44.1 kHz 스테레오 PCM16 기준 파일로 바꾸고 ② 모델로 구간을 다시 만들고 ③ 그 구간만 기준 파일에 붙입니다(경계 안쪽 0.1초 크로스페이드). 공식 가이드의 경험칙: 구간이 몇 초뿐이면 문맥이 강해서 원곡과 거의 같게 나옵니다. 처음엔 넓게 잡고 점점 줄여 보세요.

In [ ]:
#@title 인페인팅 실행
t0 = time.time()
edited = sa3.inpaint(source, EDIT_PROMPT, start=START_SECONDS, end=END_SECONDS, seed=SEED, steps=STEPS, cfg=CFG,
                     negative_prompt=NEGATIVE_PROMPT, crossfade=CROSSFADE_SECONDS, max_seconds=MAX_INPUT_SECONDS)
print('%.0f s · 최종 파일: %s' % (time.time() - t0, edited))

In [ ]:
#@title 전체 곡 비교: 청록 점선이 편집 경계
from pathlib import Path
report = json.loads((Path(edited).parent / "edit_report.json").read_text())
comparison_clips = [("Before — original", Path(report["reference"])), (f"After — {TARGET_STYLE} (CFG {CFG:g})", Path(edited))]
sa3.compare_audio(comparison_clips, region=(START_SECONDS, END_SECONDS), title="Inpainting: original vs edited",
                  output_dir=Path(edited).parent, filename="inpaint_full_comparison")

In [ ]:
#@title 편집 구간만 확대 비교
sa3.compare_audio(comparison_clips, crop=(START_SECONDS, END_SECONDS), title="Inpainting: selected interval only",
                  output_dir=Path(edited).parent, filename="inpaint_region_comparison")

In [ ]:
#@title 검증: 구간 밖은 그대로, 구간 안은 바뀌었는지
import numpy as np, soundfile as sf
reference, sr = sf.read(report["reference"], dtype="int16", always_2d=True)
result, result_sr = sf.read(edited, dtype="int16", always_2d=True)
a, b = report["start_sample"], report["end_sample"]
assert sr == result_sr == 44100 and reference.shape == result.shape
assert np.array_equal(reference[:a], result[:a])
assert np.array_equal(reference[b:], result[b:])
assert not np.array_equal(reference[a:b], result[a:b])
print("PASS: 구간 밖 완전 보존, 구간 안 실제 변경.")

## 해 보기

1. **스타일 바꾸기**: `TARGET_STYLE`을 `drum_and_bass`로 바꾸고 원본 셀부터 다시 실행합니다. 스펙트로그램에서 리듬 밀도가 어떻게 달라지나요?
2. **CFG 올리기**: `CFG = 6.0`이면 변화가 커지는 대신 음량과 왜곡도 커집니다. 플레이어의 RMS·Peak를 비교해 보세요.
3. **오디오 → 오디오 변형**: 아래 셀은 구간이 아니라 곡 전체를 다른 스타일로 바꿉니다. `NOISE_LEVEL`이 클수록 원곡에서 멀어집니다. 0.3이면 원곡 골격이 남고, 0.6이면 스타일이 바뀌고, 1.0이면 처음부터 새로 만듭니다.

In [ ]:
#@title 오디오 → 오디오 변형 (곡 전체)
RUN_VARIATION = False  #@param {type:"boolean"}
VARIATION_PROMPT = "Instrumental warm jazz trio, electric piano, upright bass, brushed drums, no vocals"  #@param {type:"string"}
NOISE_LEVEL = 0.6  #@param {type:"slider", min:0.1, max:1.0, step:0.1}
if RUN_VARIATION:
    varied = sa3.variation(report["reference"], VARIATION_PROMPT, noise_level=NOISE_LEVEL, seed=45)
    sa3.compare_audio([("Original", Path(report["reference"])), (f"Variation (noise {NOISE_LEVEL:g})", Path(varied))],
                      title="Audio-to-audio variation", output_dir=Path(varied).parent, filename="variation_comparison")

In [ ]:
#@title 저장: 결과 폴더를 zip으로 내려받기
import shutil
job_dir = str(Path(edited).parent)
zip_path = shutil.make_archive(job_dir, 'zip', job_dir)
print('저장:', zip_path, '(edited.wav가 최종 편집 파일, inpaint_full_comparison.html은 그림과 소리를 같이 봄)')
if IN_COLAB and not OUT_DIR.startswith('/content/drive'):
    from google.colab import files; files.download(zip_path)